<a href="https://colab.research.google.com/github/Mebay821/health-hackathon-project/blob/main/diabetes_risk_screening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","DiabetesPedigree","Age","Outcome"]
df = pd.read_csv(URL, names=cols)

print(df.shape)
print(df["Outcome"].mean())   # share of people with diabetes
df.head()

(768, 9)
0.3489583333333333


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigree,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
for c in ["Glucose","BloodPressure","BMI"]:
    df[c] = df[c].replace(0, np.nan)
    df[c] = df[c].fillna(df[c].median())

In [ ]:
X = df[["Age","BMI","BloodPressure","Glucose"]]
y = df["Outcome"]

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42)

In [ ]:
def report(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    print(f"{name}: accuracy {accuracy_score(y_true,y_pred):.1%}, recall {recall_score(y_true,y_pred):.1%}, "
          f"missed cases {fn}, false alarms {fp}")

# Baseline: everyone is "no diabetes"
report("Baseline", y_val, np.zeros(len(y_val)))

# First model
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)
report("Model", y_val, model.predict(X_val))

Baseline: accuracy 64.9%, recall 0.0%, missed cases 54, false alarms 0
Model: accuracy 74.7%, recall 74.1%, missed cases 14, false alarms 25


In [ ]:
probs = model.predict_proba(X_val)[:,1]
for t in [0.5, 0.4, 0.3, 0.2]:
    report(f"threshold {t}", y_val, (probs >= t).astype(int))

threshold 0.5: accuracy 74.7%, recall 74.1%, missed cases 14, false alarms 25
threshold 0.4: accuracy 69.5%, recall 75.9%, missed cases 13, false alarms 34
threshold 0.3: accuracy 68.2%, recall 87.0%, missed cases 7, false alarms 42
threshold 0.2: accuracy 61.7%, recall 94.4%, missed cases 3, false alarms 56


In [ ]:
X = df[["Age","BMI","BloodPressure","Glucose"]]
y = df["Outcome"]

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42)

print(len(X_train), len(X_val), len(X_test))

460 154 154


In [ ]:
def report(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    print(f"{name}: accuracy {accuracy_score(y_true,y_pred):.1%}, recall {recall_score(y_true,y_pred):.1%}, "
          f"missed cases {fn}, false alarms {fp}")

report("Baseline", y_val, np.zeros(len(y_val)))

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)
report("Model", y_val, model.predict(X_val))

Baseline: accuracy 64.9%, recall 0.0%, missed cases 54, false alarms 0
Model: accuracy 74.7%, recall 74.1%, missed cases 14, false alarms 25


In [ ]:
probs = model.predict_proba(X_val)[:,1]
for t in [0.5, 0.4, 0.3, 0.2]:
    report(f"threshold {t}", y_val, (probs >= t).astype(int))

threshold 0.5: accuracy 74.7%, recall 74.1%, missed cases 14, false alarms 25
threshold 0.4: accuracy 69.5%, recall 75.9%, missed cases 13, false alarms 34
threshold 0.3: accuracy 68.2%, recall 87.0%, missed cases 7, false alarms 42
threshold 0.2: accuracy 61.7%, recall 94.4%, missed cases 3, false alarms 56


In [ ]:
t = 0.3
test_probs = model.predict_proba(X_test)[:,1]
report("Baseline (test)", y_test, np.zeros(len(y_test)))
report(f"Model, threshold {t} (test)", y_test, (test_probs >= t).astype(int))

Baseline (test): accuracy 64.9%, recall 0.0%, missed cases 54, false alarms 0
Model, threshold 0.3 (test): accuracy 66.9%, recall 87.0%, missed cases 7, false alarms 44


In [ ]:
test_pred = (test_probs >= 0.3).astype(int)
groups = pd.cut(X_test["Age"], bins=[0, 30, 45, 100], labels=["under 30", "30-45", "over 45"])

for g in ["under 30", "30-45", "over 45"]:
    mask = (groups == g).values
    n_sick = int(y_test[mask].sum())
    tn, fp, fn, tp = confusion_matrix(y_test[mask], test_pred[mask]).ravel()
    print(f"{g}: {mask.sum()} patients, {n_sick} sick, caught {tp}, missed {fn}, false alarms {fp}")

under 30: 89 patients, 18 sick, caught 14, missed 4, false alarms 27
30-45: 40 patients, 26 sick, caught 23, missed 3, false alarms 8
over 45: 25 patients, 10 sick, caught 10, missed 0, false alarms 9


In [ ]:
val_groups = pd.cut(X_val["Age"], bins=[0, 30, 45, 100], labels=["under 30", "30-45", "over 45"])
mask = (val_groups == "under 30").values

for t in [0.3, 0.25, 0.2, 0.15]:
    pred = (probs[mask] >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val[mask], pred).ravel()
    print(f"threshold {t}: caught {tp}, missed {fn}, false alarms {fp}")

threshold 0.3: caught 11, missed 3, false alarms 18
threshold 0.25: caught 12, missed 2, false alarms 21
threshold 0.2: caught 14, missed 0, false alarms 28
threshold 0.15: caught 14, missed 0, false alarms 37


In [ ]:
thr = np.where(X_test["Age"] <= 30, 0.2, 0.3)
pred_group = (test_probs >= thr).astype(int)

report("Single threshold 0.3 (test)", y_test, (test_probs >= 0.3).astype(int))
report("Per-age threshold (test)", y_test, pred_group)

mask = (X_test["Age"] <= 30).values
tn, fp, fn, tp = confusion_matrix(y_test[mask], pred_group[mask]).ravel()
print(f"Under 30 only: caught {tp}, missed {fn}, false alarms {fp}")

Single threshold 0.3 (test): accuracy 66.9%, recall 87.0%, missed cases 7, false alarms 44
Per-age threshold (test): accuracy 61.7%, recall 90.7%, missed cases 5, false alarms 54
Under 30 only: caught 16, missed 2, false alarms 37


In [ ]:
def risk_check(age, bmi, blood_pressure, glucose):
    x = pd.DataFrame([[age, bmi, blood_pressure, glucose]], columns=["Age","BMI","BloodPressure","Glucose"])
    p = model.predict_proba(x)[0, 1]
    cut = 0.2 if age <= 30 else 0.3
    if p >= cut:
        return f"HIGH risk (score {p:.0%}). Refer for a blood test."
    elif p >= 0.2:
        return f"MEDIUM risk (score {p:.0%}). Recheck in a few months, or refer if symptoms."
    return f"LOW risk (score {p:.0%}). No action needed now."

print(risk_check(25, 22, 70, 95))
print(risk_check(52, 34, 85, 160))

LOW risk (score 6%). No action needed now.
HIGH risk (score 87%). Refer for a blood test.


In [ ]:
# 1. Final Test Comparison: Single Threshold vs Per-Group Threshold
thr = np.where(X_test["Age"] <= 30, 0.2, 0.3)
pred_group = (test_probs >= thr).astype(int)

report("Single threshold 0.3 (test)", y_test, (test_probs >= 0.3).astype(int))
report("Per-group threshold (test)", y_test, pred_group)

# 2. Final Risk Check Function (Great for your demo/testing)
def risk_check(age, bmi, blood_pressure, glucose):
    # Predict probability using your trained model
    input_data = pd.DataFrame([[age, bmi, blood_pressure, glucose]],
                              columns=["Age", "BMI", "BloodPressure", "Glucose"])
    prob = model.predict_proba(input_data)[:, 1][0]

    # Determine threshold based on age group
    threshold = 0.2 if age <= 30 else 0.3

    # Assign risk level
    if prob >= threshold:
        return f"High Risk ({prob:.1%}): Recommend immediate doctor referral & blood test."
    elif prob >= (threshold - 0.1):
        return f"Medium Risk ({prob:.1%}): Suggest lifestyle review and follow-up."
    else:
        return f"Low Risk ({prob:.1%}): Normal indicators."

# Quick test of the function
print(risk_check(age=25, bmi=28.5, blood_pressure=72, glucose=140))


Single threshold 0.3 (test): accuracy 66.9%, recall 87.0%, missed cases 7, false alarms 44
Per-group threshold (test): accuracy 61.7%, recall 90.7%, missed cases 5, false alarms 54
High Risk (44.8%): Recommend immediate doctor referral & blood test.


In [60]:
import re, tempfile
import numpy as np
import pandas as pd
import gradio as gr
from gtts import gTTS
from transformers import pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# --- 1. AUTOMATIC MODEL TRAINING ---
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","DiabetesPedigree","Age","Outcome"]
df = pd.read_csv(URL, names=cols)
for c in ["Glucose","BloodPressure","BMI"]:
    df[c] = df[c].replace(0, np.nan)
    df[c] = df[c].fillna(df[c].median())

X = df[["Age","BMI","BloodPressure","Glucose"]]
y = df["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

# --- 2. LOCALIZATION DICTIONARIES ---
LANGS = {
    "English": "en", "አማርኛ (Amharic)": "am", "Español": "es",
    "Français": "fr", "العربية": "ar", "中文 (Mandarin)": "zh-CN",
}

UI_TEXT = {
    "en": {
        "title": "sugarSpot: Intelligent Diabetes Risk Screening",
        "subtitle": "Clinical screening tool with strict validation guardrails.",
        "lang_label": "Select Language (UI & Voice)",
        "mic_label": "Hands-free Voice Input: Press record and speak your numbers",
        "age_label": "Age (1 - 120 years)",
        "bmi_label": "Body Mass Index (BMI: 10 - 70)",
        "bp_label": "Blood Pressure / Diastolic (40 - 150)",
        "glu_label": "Fasting Glucose (mg/dL: 50 - 400)",
        "btn_text": "Check Risk Assessment",
        "result_label": "Assessment Summary",
        "voice_label": "Spoken Voice Output",
        "wizard_tab": "🤖 Smart Estimator Wizard",
        "form_tab": "📋 Screening Form",
        "wizard_title": "🔍 Don't know your numbers? Answer these quick questions",
        "gender_label": "Gender",
        "height_label": "Height (cm)",
        "weight_label": "Weight (kg)",
        "calc_btn": "Estimate & Fill Form"
    },
    "am": {
        "title": "ሹገርስፖት: የስኳር በሽታ ቅድመ ምርመራ መሣሪያ",
        "subtitle": "ደህንነቱ የተጠበቀ የጤና ማጣሪያ መሣሪያ።",
        "lang_label": "ቋንቋ ይምረጡ",
        "mic_label": "በድምጽ ለመሙላት ይጫኑ እና ቁጥሮችን ይናገሩ",
        "age_label": "ዕድሜ (1 - 120 ዓመት)",
        "bmi_label": "የሰውነት ክብደት መለኪያ (BMI: 10 - 70)",
        "bp_label": "የደም ግፊት / ዝቅተኛው (40 - 150)",
        "glu_label": "የስኳር መጠን (mg/dL: 50 - 400)",
        "btn_text": "አደጋውን ይመርምሩ",
        "result_label": "የምርመራ ማጠቃለያ",
        "voice_label": "የድምጽ ውጤት",
        "wizard_tab": "🤖 ስማርት ረዳት",
        "form_tab": "📋 የምርመራ ቅጽ",
        "wizard_title": "🔍 ቁጥሮቹን አያውቁም? እነዚህን አጭር ጥያቄዎች ይመልሱ",
        "gender_label": "ጾታ",
        "height_label": "ቁመት (ሴ.ሜ)",
        "weight_label": "ክብደት (ኪ.ግ)",
        "calc_btn": "እሴቶችን ገምተው ቅጹን ይሙሉ"
    },
    "es": {"title": "sugarSpot", "subtitle": "Detección de diabetes", "lang_label": "Idioma", "mic_label": "Voz", "age_label": "Edad", "bmi_label": "IMC", "bp_label": "Presión", "glu_label": "Glucosa", "btn_text": "Verificar", "result_label": "Resultado", "voice_label": "Voz", "wizard_tab": "Asistente", "form_tab": "Formulario", "wizard_title": "Estimación", "gender_label": "Género", "height_label": "Altura", "weight_label": "Peso", "calc_btn": "Calcular"},
    "fr": {"title": "sugarSpot", "subtitle": "Dépistage", "lang_label": "Langue", "mic_label": "Voix", "age_label": "Âge", "bmi_label": "IMC", "bp_label": "Pression", "glu_label": "Glucose", "btn_text": "Vérifier", "result_label": "Résultat", "voice_label": "Voix", "wizard_tab": "Assistant", "form_tab": "Formulaire", "wizard_title": "Estimation", "gender_label": "Genre", "height_label": "Taille", "weight_label": "Poids", "calc_btn": "Calculer"},
    "ar": {"title": "sugarSpot", "subtitle": "فحص السكري", "lang_label": "اللغة", "mic_label": "صوت", "age_label": "العمر", "bmi_label": "مؤشر الكتلة", "bp_label": "الضغط", "glu_label": "السكر", "btn_text": "فحص", "result_label": "النتيجة", "voice_label": "صوت", "wizard_tab": "مساعد", "form_tab": "استمارة", "wizard_title": "تقدير", "gender_label": "الجنس", "height_label": "الطول", "weight_label": "الوزن", "calc_btn": "احسب"},
    "zh-CN": {"title": "sugarSpot", "subtitle": "糖尿病筛查", "lang_label": "语言", "mic_label": "语音", "age_label": "年龄", "bmi_label": "BMI", "bp_label": "血压", "glu_label": "血糖", "btn_text": "检查", "result_label": "结果", "voice_label": "语音", "wizard_tab": "助手", "form_tab": "表单", "wizard_title": "估算", "gender_label": "性别", "height_label": "身高", "weight_label": "体重", "calc_btn": "计算"}
}

OUT_TEXT = {
    "en": {
        "high": "High risk detected. Immediate clinical confirmation required.",
        "medium": "Medium risk. Recommend lifestyle review and follow-up.",
        "low": "Low risk based on provided values.",
        "score": "Calculated risk probability: {p} percent.",
        "disc": "Screening tool with strict clinical range filters. Not a final medical diagnosis.",
        "invalid": "⚠️ Invalid clinical values detected! BMI must be between 10-70, Blood Pressure 40-150, and Glucose 50-400. Please correct the numbers."
    },
    "am": {
        "high": "ከፍተኛ ስጋት ተገኝቷል። ወዲያውኑ የህክምና ማረጋገጫ ያስፈልጋል።",
        "medium": "መካከለኛ ስጋት። የአኗኗር ዘይቤ ማስተካከያ እና ክትትል ይመከራል።",
        "low": "በቀረቡት መረጃዎች መሰረት ዝቅተኛ ስጋት።",
        "score": "የስጋት መቶኛ፦ {p} በመቶ።",
        "disc": "ጥብቅ የህክምና ክልል ማጣሪያ ያለው የቅድመ ምርመራ መሣሪያ። የመጨረሻ የምርመራ ማረጋገጫ አይደለም።",
        "invalid": "⚠️ የተሳሳቱ የጤና መረጃዎች ገብተዋል! እባክዎ ትክክለኛ ቁጥሮችን ያስገቡ (ቢኤምአይ 10-70፣ ግፊት 40-150፣ ግሉኮስ 50-400)።"
    },
    "es": {"high": "Riesgo alto.", "medium": "Riesgo medio.", "low": "Riesgo bajo.", "score": "Puntuación: {p}%.", "disc": "Cribado.", "invalid": "⚠️ ¡Valores clínicos inválidos! Verifique los rangos permitidos."},
    "fr": {"high": "Risque élevé.", "medium": "Risque moyen.", "low": "Risque faible.", "score": "Score : {p}%.", "disc": "Dépistage.", "invalid": "⚠️ Valeurs invalides ! Vérifiez les plages de données."},
    "ar": {"high": "خطر مرتفع.", "medium": "خطر متوسط.", "low": "خطر منخفض.", "score": "الدرجة: {p}%.", "disc": "فحص.", "invalid": "⚠️ قيم غير صالحة! يرجى التحقق من الأرقام."},
    "zh-CN": {"high": "高风险。", "medium": "中风险。", "low": "低风险。", "score": "评分：{p}%。", "disc": "筛查。", "invalid": "⚠️ 数值无效！请检查输入的健康指标。"}
}

# --- STRICT CLINICAL RANGES TO PREVENT ABSURD INPUTS ---
# Format: (Min, Max) for [Age, BMI, BloodPressure, Glucose]
STRICT_RANGES = [(1, 120), (10, 70), (40, 150), (50, 400)]

def assess(age, bmi, bp, glucose):
    x = pd.DataFrame([[age, bmi, bp, glucose]], columns=["Age", "BMI", "BloodPressure", "Glucose"])
    p = model.predict_proba(x)[0, 1]
    cut = 0.2 if age <= 30 else 0.3
    if p >= cut: return "high", p
    if p >= 0.2: return "medium", p
    return "low", p

def to_float(s):
    try: return float(str(s).strip().replace(",", "."))
    except ValueError: return None

def speak_to_file(text, code):
    try:
        path = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
        clean_text = str(text).replace("{p}", "")
        gTTS(text=clean_text, lang=code).save(path)
        return path
    except Exception as e:
        print("Voice generation failed:", e)
        return None

def run(age, bmi, bp, glucose, lang_label):
    code = LANGS[lang_label]
    T = OUT_TEXT[code]
    vals = [to_float(v) for v in (age, bmi, bp, glucose)]

    # Check if any value failed parsing OR falls outside strict clinical boundaries
    ok = all(v is not None for v in vals) and all(lo <= v <= hi for v, (lo, hi) in zip(vals, STRICT_RANGES))

    if not ok:
        msg = T["invalid"]
    else:
        level, p = assess(*vals)
        msg = f'{T[level]} {T["score"].format(p=round(p * 100))} {T["disc"]}'
    return msg, speak_to_file(msg, code)

asr = pipeline("automatic-speech-recognition", model="openai/whisper-base")

def parse_speech(text):
    t = text.lower()
    def find(words):
        for w in words:
            m = re.search(w + r"\D{0,15}(\d+(?:\.\d+)?)", t)
            if m: return m.group(1)
        return ""
    return (find(["age"]), find(["bmi", "b m i", "body mass"]), find(["blood pressure", "pressure"]), find(["glucose", "sugar"]))

def listen(audio_path):
    if audio_path is None: return "", "", "", "", "Nothing recorded."
    text = asr(audio_path)["text"]
    a, b, c, d = parse_speech(text)
    return a, b, c, d, f"Heard: {text}"

def detailed_estimate_wizard(gender, age_approx, has_hypertension, feels_thirsty, height_cm, weight_kg):
    try:
        h = float(height_cm) / 100
        w = float(weight_kg)
        bmi_val = round(w / (h ** 2), 1)
        bp_val = 90 if has_hypertension == "Yes" else (75 if gender == "Female" else 80)
        age_num = float(age_approx) if age_approx else 30
        glu_val = 140 if feels_thirsty == "Yes (Frequent thirst/fatigue)" else (95 if age_num < 40 else 115)
        status_msg = f"Successfully estimated! BMI: {bmi_val}, BP: {bp_val}, Glucose: {glu_val}"
        return bmi_val, bp_val, glu_val, status_msg
    except Exception as e:
        return "", "", "", "Please provide valid height and weight numbers."

def update_ui_labels(lang_label):
    code = LANGS[lang_label]
    t = UI_TEXT[code]
    return (
        gr.update(value=f"# {t['title']}\n*{t['subtitle']}*"),
        gr.update(label=t['lang_label']),
        gr.update(label=t['mic_label']),
        gr.update(label=t['age_label'], placeholder="e.g. 45"),
        gr.update(label=t['bmi_label'], placeholder="e.g. 27.5"),
        gr.update(label=t['bp_label'], placeholder="e.g. 80"),
        gr.update(label=t['glu_label'], placeholder="e.g. 95"),
        gr.update(value=t['btn_text']),
        gr.update(label=t['result_label']),
        gr.update(label=t['voice_label']),
        gr.update(label=t['form_tab']),
        gr.update(label=t['wizard_tab']),
        gr.update(value=f"### {t['wizard_title']}"),
        gr.update(label=t['gender_label']),
        gr.update(label="Approximate Age"),
        gr.update(label=t['height_label']),
        gr.update(label=t['weight_label']),
        gr.update(label="Do you have a history of high blood pressure?"),
        gr.update(label="Are you experiencing unusual thirst or fatigue?"),
        gr.update(value=t['calc_btn']),
    )

CSS = """
body { background: #f8fafc; }
.big-btn { font-size: 1.4rem !important; background: linear-gradient(135deg, #2563eb, #1d4ed8) !important; color: white !important; border-radius: 12px !important; }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", text_size="lg"), css=CSS, title="sugarSpot") as demo:
    header_md = gr.Markdown("# sugarSpot: Intelligent Diabetes Risk Screening\n*Clinical screening tool with strict validation guardrails.*")

    with gr.Row():
        lang = gr.Dropdown(list(LANGS), value="English", label="Select Language (UI & Voice)")

    with gr.Tabs() as tabs:
        with gr.TabItem("📋 Screening Form", id="form_tab") as tab_form:
            mic = gr.Audio(sources=["microphone"], type="filepath", label='Hands-free Voice Input: Press record and speak numbers')
            heard = gr.Textbox(label="Microphone Transcript", interactive=False)

            with gr.Row():
                age = gr.Textbox(label="Age (1 - 120 years)", placeholder="e.g. 45")
                bmi = gr.Textbox(label="Body Mass Index (BMI: 10 - 70)", placeholder="e.g. 27.5")
                bp = gr.Textbox(label="Blood Pressure / Diastolic (40 - 150)", placeholder="e.g. 80")
                glu = gr.Textbox(label="Fasting Glucose (mg/dL: 50 - 400)", placeholder="e.g. 95")

            btn = gr.Button("Check Risk Assessment", variant="primary", elem_classes=["big-btn"])
            out = gr.Textbox(label="Assessment Summary", interactive=False)
            voice = gr.Audio(label="Spoken Voice Output", autoplay=True)

        with gr.TabItem("🤖 Smart Estimator Wizard", id="wizard_tab") as tab_wizard:
            wizard_header = gr.Markdown("### Don't know your numbers? Answer these quick guided questions.")
            with gr.Row():
                wizard_gender = gr.Radio(["Female", "Male"], label="Gender", value="Female")
                wizard_age = gr.Textbox(label="Approximate Age", placeholder="35")
            with gr.Row():
                wizard_height = gr.Textbox(label="Height (cm)", placeholder="165")
                wizard_weight = gr.Textbox(label="Weight (kg)", placeholder="70")
            with gr.Row():
                wizard_htn = gr.Radio(["No", "Yes"], label="Do you have a history of high blood pressure?", value="No")
                wizard_symptoms = gr.Radio(["No", "Yes (Frequent thirst/fatigue)"], label="Are you experiencing unusual thirst or fatigue?", value="No")

            wizard_btn = gr.Button("Estimate & Fill Form", variant="secondary")
            wizard_status = gr.Textbox(label="Wizard Status", interactive=False)

    wizard_btn.click(
        detailed_estimate_wizard,
        inputs=[wizard_gender, wizard_age, wizard_htn, wizard_symptoms, wizard_height, wizard_weight],
        outputs=[bmi, bp, glu, wizard_status]
    )

    lang.change(
        update_ui_labels,
        inputs=[lang],
        outputs=[
            header_md, lang, mic, age, bmi, bp, glu, btn, out, voice,
            tab_form, tab_wizard, wizard_header, wizard_gender, wizard_age,
            wizard_height, wizard_weight, wizard_htn, wizard_symptoms, wizard_btn
        ]
    )

    inputs = [age, bmi, bp, glu, lang]
    btn.click(run, inputs, [out, voice])
    mic.stop_recording(listen, mic, [age, bmi, bp, glu, heard]).then(run, inputs, [out, voice])

demo.launch(share=True)

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

/tmp/ipykernel_2309/3643541836.py:202: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", text_size="lg"), css=CSS, title="sugarSpot") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7055d0a168e925c9b6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
